# Circuitos Multi-Qubit

## Objetivo
Construir circuitos com múltiplos qubits e portas controladas.

## Portas Abordadas
- **CNOT (CX):** Controlled-NOT
- **CZ:** Controlled-Z
- **SWAP:** Troca estados de dois qubits
- **Toffoli (CCX):** Controlled-Controlled-NOT

## Referências
- Livro: Capítulo 10
- [Qiskit Textbook](https://qiskit.org/textbook/ch-gates/multiple-qubits-entangled-states.html)

In [ ]:
# Imports
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_state_qsphere
import matplotlib.pyplot as plt

## 1. Porta CNOT (Controlled-NOT)

A CNOT inverte o qubit alvo SE o qubit de controle for |1⟩.

In [ ]:
# CNOT com controle em |0⟩ - não inverte o alvo
qc1 = QuantumCircuit(2)
qc1.cx(0, 1)  # CNOT: controle=q0, alvo=q1
print("CNOT com controle |0⟩:")
print(qc1.draw())
state1 = Statevector.from_instruction(qc1)
print(f"Estado: {state1}")
print("→ Alvo permanece |0⟩ pois controle é |0⟩\n")

# CNOT com controle em |1⟩ - inverte o alvo
qc2 = QuantumCircuit(2)
qc2.x(0)       # Coloca controle em |1⟩
qc2.cx(0, 1)   # CNOT agora inverte o alvo
print("CNOT com controle |1⟩:")
print(qc2.draw())
state2 = Statevector.from_instruction(qc2)
print(f"Estado: {state2}")
print("→ Alvo inverte para |1⟩ pois controle é |1⟩")

## 2. Criando Emaranhamento com CNOT

H + CNOT = Estado de Bell!

In [ ]:
# Criando Estado de Bell |Φ+⟩ = (|00⟩ + |11⟩) / √2
qc_bell = QuantumCircuit(2)
qc_bell.h(0)       # Coloca q0 em superposição
qc_bell.cx(0, 1)   # Emaranha q0 e q1

print("Circuito para Estado de Bell |Φ+⟩:")
print(qc_bell.draw())

state_bell = Statevector.from_instruction(qc_bell)
print(f"\nStatevector: {state_bell}")
print("\n→ Este é um estado EMARANHADO!")
print("→ Se medirmos q0 e obtermos |0⟩, q1 também será |0⟩")
print("→ Se medirmos q0 e obtermos |1⟩, q1 também será |1⟩")

# Visualizar
display(plot_state_qsphere(state_bell))

## 3. Porta CZ (Controlled-Z)

In [ ]:
# Porta CZ - adiciona fase -1 apenas se AMBOS qubits forem |1⟩
qc_cz = QuantumCircuit(2)
qc_cz.h(0)
qc_cz.h(1)
qc_cz.cz(0, 1)

print("CZ em superposição:")
print(qc_cz.draw())

state_cz = Statevector.from_instruction(qc_cz)
print(f"\nStatevector: {state_cz}")
print("\n→ Apenas |11⟩ tem fase invertida (-0.5 em vez de +0.5)")
print("→ CZ é simétrica: cz(0,1) = cz(1,0)")

## 4. Porta SWAP

In [ ]:
# SWAP - troca estados de dois qubits
qc_swap = QuantumCircuit(2)
qc_swap.x(1)       # Criar estado |01⟩ (q0=0, q1=1)

print("Estado inicial |01⟩:")
state_before = Statevector.from_instruction(qc_swap)
print(f"Antes do SWAP: {state_before}")

qc_swap.swap(0, 1)  # Trocar q0 ↔ q1

print("\nCircuito com SWAP:")
print(qc_swap.draw())

state_after = Statevector.from_instruction(qc_swap)
print(f"\nDepois do SWAP: {state_after}")
print("→ Estado trocou de |01⟩ para |10⟩")

## 5. Implementar SWAP com CNOTs

SWAP pode ser decomposto em 3 CNOTs.

In [ ]:
# SWAP implementado com 3 CNOTs
from qiskit.quantum_info import Operator

# SWAP nativo
qc_swap_native = QuantumCircuit(2)
qc_swap_native.swap(0, 1)

# SWAP com CNOTs: CNOT(0,1) → CNOT(1,0) → CNOT(0,1)
qc_swap_cnot = QuantumCircuit(2)
qc_swap_cnot.cx(0, 1)
qc_swap_cnot.cx(1, 0)
qc_swap_cnot.cx(0, 1)

print("SWAP com 3 CNOTs:")
print(qc_swap_cnot.draw())

# Verificar equivalência
op_native = Operator(qc_swap_native)
op_cnot = Operator(qc_swap_cnot)
print(f"\nSWAP nativo = 3 CNOTs? {op_native.equiv(op_cnot)}")

## 6. Porta Toffoli (CCX)

Inverte o alvo se AMBOS os controles forem |1⟩.

In [ ]:
# Porta Toffoli (CCX) - CNOT duplo controle
# Inverte alvo apenas se AMBOS controles forem |1⟩

# Teste 1: Controles |00⟩
qc_tof1 = QuantumCircuit(3)
qc_tof1.ccx(0, 1, 2)
s1 = Statevector.from_instruction(qc_tof1)
print(f"|00⟩ + Toffoli → {s1} (alvo permanece |0⟩)")

# Teste 2: Controles |10⟩
qc_tof2 = QuantumCircuit(3)
qc_tof2.x(0)
qc_tof2.ccx(0, 1, 2)
s2 = Statevector.from_instruction(qc_tof2)
print(f"|10⟩ + Toffoli → {s2} (alvo permanece |0⟩)")

# Teste 3: Controles |01⟩
qc_tof3 = QuantumCircuit(3)
qc_tof3.x(1)
qc_tof3.ccx(0, 1, 2)
s3 = Statevector.from_instruction(qc_tof3)
print(f"|01⟩ + Toffoli → {s3} (alvo permanece |0⟩)")

# Teste 4: Controles |11⟩ - ÚNICO CASO QUE INVERTE
qc_tof4 = QuantumCircuit(3)
qc_tof4.x(0)
qc_tof4.x(1)
qc_tof4.ccx(0, 1, 2)
s4 = Statevector.from_instruction(qc_tof4)
print(f"|11⟩ + Toffoli → {s4} (alvo INVERTE para |1⟩)")

print("\nCircuito Toffoli:")
qc_tof4_draw = QuantumCircuit(3)
qc_tof4_draw.ccx(0, 1, 2)
print(qc_tof4_draw.draw())

## 7. Circuito de Paridade

Crie um circuito que computa a paridade de 2 qubits.

In [ ]:
# Circuito de Paridade
# q2 = q0 XOR q1 (1 se paridade ímpar)

def parity_circuit(q0_val: int, q1_val: int):
    """Computa paridade de dois qubits."""
    qc = QuantumCircuit(3)
    
    # Configurar valores de entrada
    if q0_val:
        qc.x(0)
    if q1_val:
        qc.x(1)
    
    # Paridade: dois CNOTs para q2
    qc.cx(0, 2)  # q2 = q0
    qc.cx(1, 2)  # q2 = q0 XOR q1
    
    return qc

print("Circuito de Paridade:")
qc_par = QuantumCircuit(3)
qc_par.cx(0, 2)
qc_par.cx(1, 2)
print(qc_par.draw())

print("\nTabela verdade:")
for q0 in [0, 1]:
    for q1 in [0, 1]:
        qc = parity_circuit(q0, q1)
        state = Statevector.from_instruction(qc)
        # Encontrar estado medido
        probs = state.probabilities_dict()
        result = list(probs.keys())[0]
        parity = result[0]  # q2 é o primeiro caractere (little-endian invertido)
        print(f"  q0={q0}, q1={q1} → paridade={parity} (XOR={q0^q1})")

## 8. Conclusão

Responda:
- Por que CNOT é tão importante?
- O que significa "emaranhamento" na prática?
- Quantos CNOTs são necessários para um SWAP?

**Resposta:**

1. **Por que CNOT é tão importante?**
   - É a porta fundamental para criar **emaranhamento** entre qubits
   - Junto com portas de um qubit, forma um conjunto **universal** (qualquer operação pode ser construída)
   - É a base para correção de erros quânticos e comunicação quântica

2. **O que significa "emaranhamento" na prática?**
   - Dois qubits emaranhados estão **correlacionados** de forma impossível classicamente
   - Medir um qubit **instantaneamente** determina o resultado do outro
   - Não é possível descrever o estado de cada qubit separadamente
   - Base para teletransporte quântico e criptografia quântica

3. **Quantos CNOTs são necessários para um SWAP?**
   - **3 CNOTs**: CNOT(a,b) → CNOT(b,a) → CNOT(a,b)
   - Esta é a decomposição mínima do SWAP em portas de 2 qubits